# Main Experiment Grid Search - V4 (With VALIDATION + TEST Metrics)

## 🔧 Version 4.0 Changes (2026-01-27)

### ✅ New: Save VALIDATION + TEST Performance
**Previous version (v3)**:
- Saved only TEST performance (RMSE, MAD, Precision, Recall)
- VALIDATION performance (used for alpha selection) was NOT saved
- Cannot verify why specific alpha was chosen
- Cannot analyze validation→test gap for overfitting detection

**This version (v4)**:
- **Saves VALIDATION metrics**: MSE, RMSE, Precision, Recall (alpha selection basis)
- **Saves TEST metrics**: RMSE, MAD, Precision, Recall (generalization)
- **Saves regularization info**: penalty, regularized_score
- **Enables validation-test gap analysis** for overfitting detection

### 🔄 V3 Features (Still Included)
- Regularization penalty: λ × |α - 1.0| to prevent overfitting
- Alpha range: 0.0 ~ 3.0 (prevents extreme values)
- Finer grid search: step 0.25 for better precision

### 📂 Data Files Required
For each fold_XX/:
- `train_inner.csv` - Training data for KNN (created by create_train_validation_split_0113.ipynb)
- `validation.csv` - Validation data for α optimization
- `train.csv` - Full train (for final evaluation with optimal α)
- `test.csv` - Test data (final evaluation ONLY)

### 📊 Previous Results (V2 - Overfitted)
Archived in: `results/archived_overfitted/`
- See README.md for overfitting analysis

## Step 1: Imports and Setup

In [9]:
import os
import time
import json
import shutil
import numpy as np
import pandas as pd
from typing import Tuple, Dict, List, Optional
from datetime import datetime

print("✅ Imports loaded")
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✅ Imports loaded
Timestamp: 2026-01-28 18:24:13


## Step 2: Configuration and Incremental Strategy Settings

In [10]:
# ========================================
# INCREMENTAL EXPERIMENT STRATEGY
# ========================================
AUTO_MERGE_WITH_EXISTING = False  # Disable for fresh experiment
RUN_ONLY_NEW_METHODS = False       # Run all methods
CREATE_BACKUP = True               # Backup existing files before merge

# ========================================
# FOLD SELECTION - ALL FOLDS (RE-RUN DUE TO STRUCTURE CHANGE)
# ========================================
FOLDS_TO_RUN = [8,9,10]  # All 10 folds
# Changed: Alpha history now includes TopN dimension (10 values per alpha)
# Previous results incompatible - full re-run required

# ========================================
# GRID SEARCH RANGES
# ========================================
K_RANGE = range(10, 101, 10)  # 10, 20, 30, ..., 100
TOPN_RANGE = [5, 10, 15, 20, 25, 30, 35, 40, 45, 50]  # 10 values

# ========================================
# METRICS SETTINGS
# ========================================
RELEVANCE_THRESHOLD = 4.0  # Rating >= 4.0 considered as "liked"

# ========================================
# ALPHA OPTIMIZATION SETTINGS (V3: WITH REGULARIZATION)
# ========================================
ALPHA_SEARCH_K = 20  # DEPRECATED - now using per-K optimization
# Alpha search now uses ALL TopN values from TOPN_RANGE for comprehensive history

# V3 UPDATE: Reduced range to prevent overfitting
ALPHA_COARSE_GRID = np.arange(0.0, 10.5, 0.5)  # Coarse search grid (0.0 to 10.0, step 0.5)
ALPHA_FINE_RADIUS = 0.5  # Search radius around best coarse
ALPHA_FINE_STEP = 0.05    # Fine search step size

# V3: Regularization to prevent extreme alpha values
REGULARIZATION_LAMBDA = 0.002  # Penalty weight for deviation from α=1.0
# Regularized score = MSE + λ × |α - 1.0|
# This encourages α close to 1.0 unless there's strong evidence for different value

# V4 NEW: Store validation performance for analysis
STORE_VALIDATION_METRICS = True  # Save validation performance alongside test

# ========================================
# METHODS TO TEST - All 17 methods
# ========================================
METHODS_TO_TEST = [
    'pcc', 'cosine', 'msd', 'jmsd', 'jaccard', 'spcc', 'kendall_tau_b',
    'acos', 'src', 'cpcc', 'ari', 'ami', 'chebyshev', 'euclidean', 
    'manhattan', 'ipwr', 'itr'
 ]

print("Configuration:")
print(f"  Incremental mode:  {AUTO_MERGE_WITH_EXISTING}")
print(f"  Run only new:      {RUN_ONLY_NEW_METHODS}")
print(f"  Create backup:     {CREATE_BACKUP}")
print(f"  Folds to run:      {FOLDS_TO_RUN} ({len(FOLDS_TO_RUN)} folds)")
print(f"  K Range:           {list(K_RANGE)}")
print(f"  TopN Range:        {list(TOPN_RANGE)}")
print(f"  Methods requested: {len(METHODS_TO_TEST)} methods")
print(f"\n🆕 V3 REGULARIZATION SETTINGS:")
print(f"  Alpha Coarse:      {len(ALPHA_COARSE_GRID)} values (0.0 to 3.0, step 0.25)")
print(f"  Alpha Fine:        ±{ALPHA_FINE_RADIUS} radius, step {ALPHA_FINE_STEP}")
print(f"  Regularization λ:  {REGULARIZATION_LAMBDA}")
print(f"  Penalty formula:   MSE + {REGULARIZATION_LAMBDA} × |α - 1.0|")

Configuration:
  Incremental mode:  False
  Run only new:      False
  Create backup:     True
  Folds to run:      [8, 9, 10] (3 folds)
  K Range:           [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
  TopN Range:        [5, 10, 15, 20, 25, 30, 35, 40, 45, 50]
  Methods requested: 17 methods

🆕 V3 REGULARIZATION SETTINGS:
  Alpha Coarse:      21 values (0.0 to 3.0, step 0.25)
  Alpha Fine:        ±0.5 radius, step 0.05
  Regularization λ:  0.002
  Penalty formula:   MSE + 0.002 × |α - 1.0|


## Step 3: Incremental Strategy Helper Functions

In [11]:
def get_latest_result_file(results_dir: str, prefix: str = 'grid_search_results_') -> Optional[str]:
    """
    Find the latest result file in the results directory.
    
    Args:
        results_dir: Path to results directory
        prefix: File prefix to search for
    
    Returns:
        Full path to latest file, or None if not found
    """
    if not os.path.exists(results_dir):
        return None
    
    files = [f for f in os.listdir(results_dir) if f.startswith(prefix) and f.endswith('.csv')]
    if not files:
        return None
    
    latest_file = sorted(files)[-1]  # Lexicographic sort works with timestamp format
    return os.path.join(results_dir, latest_file)


def get_experimental_conditions_from_file(filepath: str) -> Dict:
    """
    Extract experimental conditions from existing results file.
    
    Returns dict with:
        - K_values: list of unique K values
        - TopN_values: list of unique TopN values
        - methods: list of unique methods
        - n_experiments: total number of experiments
    """
    df = pd.read_csv(filepath)
    
    return {
        'K_values': sorted(df['K'].unique().tolist()),
        'TopN_values': sorted(df['TopN'].unique().tolist()),
        'methods': sorted(df['method'].unique().tolist()),
        'n_experiments': len(df),
        'n_methods': df['method'].nunique()
    }


def validate_experimental_consistency(
    existing_file: str,
    new_K_range: range,
    new_TopN_range: list,
    new_alpha_coarse: np.ndarray,
    new_alpha_fine_radius: float,
    new_alpha_fine_step: float,
    new_relevance_threshold: float
) -> Tuple[bool, str]:
    """
    Validate that new experimental conditions match existing ones.
    
    Returns:
        (is_valid, error_message)
        - is_valid: True if conditions match
        - error_message: Description of mismatch (empty if valid)
    """
    existing_conditions = get_experimental_conditions_from_file(existing_file)
    
    # Convert new ranges to lists for comparison
    new_K_list = list(new_K_range)
    
    errors = []
    
    # Check K_RANGE
    if new_K_list != existing_conditions['K_values']:
        errors.append(f"K_RANGE mismatch:")
        errors.append(f"  Existing: {existing_conditions['K_values']}")
        errors.append(f"  New:      {new_K_list}")
    
    # Check TOPN_RANGE
    if new_TopN_range != existing_conditions['TopN_values']:
        errors.append(f"TOPN_RANGE mismatch:")
        errors.append(f"  Existing: {existing_conditions['TopN_values']}")
        errors.append(f"  New:      {new_TopN_range}")
    
    # Note: Alpha settings and relevance threshold are not stored in grid results
    # They would need to be checked from alpha_optimization_history or metadata file
    # For now, we'll add a warning
    if errors:
        errors.append("")
        errors.append("⚠️  IMPORTANT: Also verify these settings match:")
        errors.append(f"  - ALPHA_COARSE_GRID (should be 0.0 to 10.0, step 0.5)")
        errors.append(f"  - ALPHA_FINE_RADIUS (should be {new_alpha_fine_radius})")
        errors.append(f"  - ALPHA_FINE_STEP (should be {new_alpha_fine_step})")
        errors.append(f"  - RELEVANCE_THRESHOLD (should be {new_relevance_threshold})")
    
    if errors:
        return False, "\n".join(errors)
    
    return True, ""


def get_new_methods(
    requested_methods: List[str],
    existing_file: Optional[str] = None
) -> List[str]:
    """
    Identify which methods are new (not in existing results).
    
    Args:
        requested_methods: List of methods user wants to test
        existing_file: Path to existing results file
    
    Returns:
        List of methods that need to be run
    """
    if existing_file is None or not os.path.exists(existing_file):
        return requested_methods
    
    existing_conditions = get_experimental_conditions_from_file(existing_file)
    existing_methods = set(existing_conditions['methods'])
    
    new_methods = [m for m in requested_methods if m not in existing_methods]
    
    return new_methods


def merge_results(
    new_results_df: pd.DataFrame,
    existing_file: str,
    output_file: str,
    create_backup: bool = True
) -> Dict:
    """
    Merge new results with existing results.
    
    Strategy:
        - Load existing results
        - Remove rows from existing that match (fold, method) in new results
        - Append new results
        - Sort by (fold, method, K, TopN)
        - Save to output file
    
    Args:
        new_results_df: DataFrame with new experimental results
        existing_file: Path to existing results CSV
        output_file: Path to save merged results
        create_backup: Whether to backup existing file
    
    Returns:
        Dict with merge statistics
    """
    # Create backup if requested
    if create_backup and os.path.exists(existing_file):
        backup_file = existing_file.replace('.csv', '_backup.csv')
        shutil.copy2(existing_file, backup_file)
        backup_path = backup_file
    else:
        backup_path = None
    
    # Load existing results
    existing_df = pd.read_csv(existing_file)
    
    # Get unique (fold, method) combinations in new results
    new_keys = new_results_df[['fold', 'method']].drop_duplicates()
    
    # Remove matching rows from existing data
    merged_df = existing_df.copy()
    for _, row in new_keys.iterrows():
        mask = (merged_df['fold'] == row['fold']) & (merged_df['method'] == row['method'])
        merged_df = merged_df[~mask]
    
    # Append new results
    merged_df = pd.concat([merged_df, new_results_df], ignore_index=True)
    
    # Sort by (fold, method, K, TopN)
    merged_df = merged_df.sort_values(['fold', 'method', 'K', 'TopN']).reset_index(drop=True)
    
    # Save
    merged_df.to_csv(output_file, index=False)
    
    # Statistics
    stats = {
        'existing_rows': len(existing_df),
        'existing_methods': existing_df['method'].nunique(),
        'new_rows': len(new_results_df),
        'new_methods': new_results_df['method'].nunique(),
        'merged_rows': len(merged_df),
        'merged_methods': merged_df['method'].nunique(),
        'removed_rows': len(existing_df) - len(merged_df) + len(new_results_df),
        'backup_file': backup_path
    }
    
    return stats


def save_merge_metadata(
    results_dir: str,
    merge_stats: Dict,
    experimental_conditions: Dict,
    timestamp: str
):
    """
    Save metadata about the merge operation.
    """
    metadata = {
        'timestamp': timestamp,
        'merge_statistics': merge_stats,
        'experimental_conditions': experimental_conditions
    }
    
    metadata_file = os.path.join(results_dir, f"merge_metadata_{timestamp}.json")
    with open(metadata_file, 'w') as f:
        json.dump(metadata, f, indent=2)
    
    return metadata_file


print("✅ Incremental strategy helper functions defined")

✅ Incremental strategy helper functions defined


## Step 4: Data Loading Functions

In [12]:
def _load_matrix_csv(path: str) -> pd.DataFrame:
    """
    Load CSV file as (items × users) rating matrix.
    
    CRITICAL FIX (Nov 11, 2025):
    - Converts index/columns to int BEFORE sorting to ensure numerical order
    - Prevents alphabetical sorting bug (1, 10, 100 vs 1, 2, 3, ...)
    """
    df = pd.read_csv(path, low_memory=False)

    # Check if it's triple format
    lower = [str(c).lower() for c in df.columns]
    if {"item", "user", "rating"}.issubset(lower):
        def col(name):
            for c in df.columns:
                if str(c).lower() == name:
                    return c
            raise KeyError(name)
        
        i, u, r = col("item"), col("user"), col("rating")
        mat = df.pivot(index=i, columns=u, values=r)
        mat = mat.apply(pd.to_numeric, errors="coerce")
        
        # FIX: Convert to int BEFORE sorting
        mat.index = mat.index.astype(int)
        mat.columns = mat.columns.astype(int)
        mat = mat.sort_index().sort_index(axis=1)
        
        return mat

    # Otherwise assume matrix format
    first_col = df.columns[0]
    
    # Check if first column should be index
    if str(first_col).lower() in ("item", "items", "item_id", "id", "index", "unnamed: 0"):
        df = df.set_index(first_col)
    
    # Clear index/column names
    df.index.name = None
    df.columns.name = None

    # Convert to numeric
    df = df.apply(pd.to_numeric, errors="coerce")

    # FIX: Convert to int BEFORE sorting
    df.index = df.index.astype(int)
    df.columns = df.columns.astype(int)
    df = df.sort_index().sort_index(axis=1)
    
    return df

print("✅ Data loading function defined")

✅ Data loading function defined


## Step 5: KNN Prediction Function

In [13]:
def _knn_predict_removed_with_S(
    X, XX, S, K=20, include_negative=False, fallback="user_mean"
):
    """
    Predict removed entries using user-based KNN with similarity S.
    """
    X = np.asarray(X, dtype=float)
    XX = np.asarray(XX, dtype=float)
    S = np.asarray(S, dtype=float)

    n_items, n_users = XX.shape
    K_cap = min(int(K), max(0, n_users - 1))

    # Find removed entries
    removed = (~np.isnan(X)) & (np.isnan(XX))
    if not np.any(removed):
        raise ValueError("No removed entries to evaluate")

    rated_mask = ~np.isnan(XX)
    user_means = np.nanmean(XX, axis=0)
    item_means = np.nanmean(XX, axis=1)
    global_mean = float(np.nanmean(XX)) if np.isfinite(np.nanmean(XX)) else 0.0

    preds = {}
    se_sum = 0.0
    cnt = 0

    rows, cols = np.where(removed)
    for i, j in zip(rows, cols):
        # Find candidate neighbors
        cand = np.where(rated_mask[i])[0]
        cand = cand[cand != j]

        if cand.size == 0 or K_cap == 0:
            if np.isfinite(user_means[j]):
                pred = float(user_means[j])
            elif np.isfinite(item_means[i]):
                pred = float(item_means[i])
            else:
                pred = global_mean
        else:
            sims = S[j, cand]
            
            if not include_negative:
                keep = np.where(sims > 0.0)[0]
                cand = cand[keep]
                sims = sims[keep]

            if cand.size == 0:
                if np.isfinite(user_means[j]):
                    pred = float(user_means[j])
                elif np.isfinite(item_means[i]):
                    pred = float(item_means[i])
                else:
                    pred = global_mean
            else:
                K_eff = min(K_cap, cand.size)
                top = np.argsort(-sims)[:K_eff]
                nbrs = cand[top]
                w = sims[top]

                w_sum = np.sum(w)
                if not np.isfinite(w_sum) or np.isclose(w_sum, 0.0):
                    if np.isfinite(user_means[j]):
                        pred = float(user_means[j])
                    elif np.isfinite(item_means[i]):
                        pred = float(item_means[i])
                    else:
                        pred = global_mean
                else:
                    w = w / w_sum
                    pred = float(np.dot(w, XX[i, nbrs]))

        preds[(i, j)] = pred
        true_val = float(X[i, j])
        se_sum += (pred - true_val) ** 2
        cnt += 1

    mse = se_sum / cnt if cnt > 0 else np.nan
    return preds, mse

print("✅ KNN prediction function defined")

✅ KNN prediction function defined


## Step 6: Metrics Calculation Functions

In [14]:
def _combine_single_similarity(S, alpha, clip_neg=True):
    """Apply alpha exponent to similarity matrix."""
    S = np.asarray(S, dtype=float)
    S_eff = np.clip(S, 0.0, None) if clip_neg else S
    S_eff = np.power(S_eff, float(alpha))
    np.fill_diagonal(S_eff, 0.0)
    return S_eff


def rmse_mad_on_test(pred: pd.DataFrame, test: pd.DataFrame) -> Tuple[float, float]:
    """Calculate RMSE and MAD on test set."""
    P = pred.values
    T = test.values
    mask = ~np.isnan(T)
    if not mask.any():
        return np.nan, np.nan
    diff = P[mask] - T[mask]
    rmse = float(np.sqrt(np.mean(diff**2)))
    mad = float(np.mean(np.abs(diff)))
    return rmse, mad


def precision_recall_at_n(
    pred: pd.DataFrame, train: pd.DataFrame, test: pd.DataFrame,
    N: int, relevance_threshold: float = 4.0
) -> Tuple[float, float]:
    """
    Calculate Precision@N and Recall@N.
    Candidate items: not in train & observed in test
    Relevant items: test rating >= relevance_threshold
    """
    precisions, recalls = [], []
    
    for u_idx in range(pred.shape[1]):
        train_col = train.iloc[:, u_idx]
        test_col = test.iloc[:, u_idx]
        pred_col = pred.iloc[:, u_idx]
        
        cand_mask = train_col.isna() & ~test_col.isna()
        if not cand_mask.any():
            continue
        
        relevant_mask = (test_col >= relevance_threshold) & cand_mask
        n_rel = int(relevant_mask.sum())
        
        scores = pred_col[cand_mask]
        topN_items = scores.sort_values(ascending=False).head(N).index
        
        n_hit = int(relevant_mask.loc[topN_items].sum()) if len(topN_items) else 0
        
        prec_u = n_hit / min(N, int(cand_mask.sum())) if int(cand_mask.sum()) > 0 else np.nan
        rec_u = n_hit / n_rel if n_rel > 0 else np.nan
        
        if not np.isnan(prec_u): 
            precisions.append(prec_u)
        if not np.isnan(rec_u):  
            recalls.append(rec_u)
    
    precision = float(np.mean(precisions)) if precisions else np.nan
    recall = float(np.mean(recalls)) if recalls else np.nan
    return precision, recall

print("✅ Metrics calculation functions defined")

✅ Metrics calculation functions defined


## Step 7: Multi-Fold Experiment Loop with Incremental Strategy

### 🔬 Experimental Data Flow

**Critical: Two-Phase Design to Prevent Test Leakage**

#### Phase 1: Alpha Optimization (VALIDATION data)
- **Learning**: `train_inner.csv` (~72% of full data)
- **Evaluation**: `validation.csv` (~18% of full data)
- **Purpose**: Find optimal α for each (method, K) combination
- **Output**: `alpha_optimization_history_*.csv`
- **⚠️ TEST DATA NEVER USED** in this phase

#### Phase 2: Final Evaluation (TEST data)
- **Learning**: `train.csv` (~90% of full data, includes train_inner + validation)
- **Evaluation**: `test.csv` (~10% of full data)
- **Purpose**: Fair comparison of optimal α vs α=1.0 baseline
- **Output**: `grid_search_results_*.csv` with type='optimized' and type='baseline'
- **✅ BOTH USE SAME TEST DATA** for fair comparison

**Result**: No test leakage, fair comparison guaranteed!

In [15]:
print(f"\n{'='*80}")
print(f"STARTING MULTI-FOLD GRID SEARCH EXPERIMENT (INCREMENTAL MODE)")
print(f"{'='*80}")
print(f"Folds to process: {FOLDS_TO_RUN}")
print(f"Total folds: {len(FOLDS_TO_RUN)}")

# Global tracking
global_start_time = time.time()
all_folds_results = []
all_folds_alpha_history = []

for fold_num in FOLDS_TO_RUN:
    fold_id = f"fold_{fold_num:02d}"
    FOLD_DIR = f"../data/movielenz_data/{fold_id}"
    TRAIN_INNER_CSV = os.path.join(FOLD_DIR, "train_inner.csv")  # V2: Changed from train.csv
    VALIDATION_CSV = os.path.join(FOLD_DIR, "validation.csv")    # V2: Added validation
    TRAIN_CSV = os.path.join(FOLD_DIR, "train.csv")              # V2: For final evaluation
    TEST_CSV = os.path.join(FOLD_DIR, "test.csv")
    RESULTS_DIR = os.path.join("../results", fold_id)
    os.makedirs(RESULTS_DIR, exist_ok=True)
    
    print(f"\n{'='*80}")
    print(f"FOLD {fold_num:02d} / {len(FOLDS_TO_RUN)}")
    print(f"{'='*80}")
    print(f"Directory: {FOLD_DIR}")
    
    # Check if fold exists (V2: Check for train_inner and validation)
    if not os.path.exists(TRAIN_INNER_CSV) or not os.path.exists(VALIDATION_CSV) or not os.path.exists(TEST_CSV):
        print(f"⚠️  Skipping {fold_id}: Required data files not found")
        print(f"     Run create_train_validation_split_0113.ipynb first!")
        continue
    
    # ========================================
    # INCREMENTAL STRATEGY: Detect new methods
    # ========================================
    methods_to_run = METHODS_TO_TEST.copy()
    merge_mode = False
    existing_results_file = None
    
    if AUTO_MERGE_WITH_EXISTING:
        existing_results_file = get_latest_result_file(RESULTS_DIR, 'grid_search_results_')
        
        if existing_results_file:
            print(f"\n📂 Found existing results: {os.path.basename(existing_results_file)}")
            
            # Validate experimental conditions
            is_valid, error_msg = validate_experimental_consistency(
                existing_file=existing_results_file,
                new_K_range=K_RANGE,
                new_TopN_range=TOPN_RANGE,
                new_alpha_coarse=ALPHA_COARSE_GRID,
                new_alpha_fine_radius=ALPHA_FINE_RADIUS,
                new_alpha_fine_step=ALPHA_FINE_STEP,
                new_relevance_threshold=RELEVANCE_THRESHOLD
            )
            
            if not is_valid:
                print(f"\n{'='*80}")
                print(f"❌ EXPERIMENTAL CONDITION MISMATCH")
                print(f"{'='*80}")
                print(error_msg)
                print(f"\n⚠️  Cannot merge with existing results due to incompatible settings.")
                print(f"Options:")
                print(f"  1. Update your configuration to match existing experiments")
                print(f"  2. Set AUTO_MERGE_WITH_EXISTING = False to start fresh")
                print(f"  3. Delete existing results if you want to change experimental conditions")
                print(f"\nSkipping {fold_id}")
                continue
            
            print(f"✅ Experimental conditions validated - compatible with existing results")
            
            # Get new methods if requested
            if RUN_ONLY_NEW_METHODS:
                new_methods = get_new_methods(METHODS_TO_TEST, existing_results_file)
                
                if not new_methods:
                    print(f"✅ All requested methods already exist in results - skipping {fold_id}")
                    continue
                
                existing_conditions = get_experimental_conditions_from_file(existing_results_file)
                print(f"\n🔍 Method Detection:")
                print(f"  Existing methods: {existing_conditions['methods']}")
                print(f"  Requested methods: {METHODS_TO_TEST}")
                print(f"  New methods to run: {new_methods}")
                
                methods_to_run = new_methods
                merge_mode = True
            else:
                print(f"ℹ️  RUN_ONLY_NEW_METHODS disabled - will run all requested methods")
                merge_mode = True
        else:
            print(f"ℹ️  No existing results found - will run all methods from scratch")
    
    # Load data (V2: Load train_inner, validation, and test)
    print(f"\nLoading data from: {FOLD_DIR}")
    XX_train_inner = _load_matrix_csv(TRAIN_INNER_CSV)  # V2: For α optimization learning
    XX_validation = _load_matrix_csv(VALIDATION_CSV)    # V2: For α optimization evaluation
    XX_train_full = _load_matrix_csv(TRAIN_CSV)         # V2: For final evaluation
    XX_test = _load_matrix_csv(TEST_CSV)                # V2: For final evaluation ONLY
    
    print(f"✓ Train_inner data: {XX_train_inner.shape} (items × users)")
    print(f"  Observed ratings: {(~XX_train_inner.isna()).sum().sum():,}")
    print(f"✓ Validation data: {XX_validation.shape} (items × users)")
    print(f"  Observed ratings: {(~XX_validation.isna()).sum().sum():,}")
    print(f"✓ Train_full data: {XX_train_full.shape} (items × users)")
    print(f"  Observed ratings: {(~XX_train_full.isna()).sum().sum():,}")
    print(f"✓ Test data: {XX_test.shape} (items × users)")
    print(f"  Observed ratings: {(~XX_test.isna()).sum().sum():,}")
    
    # Run grid search for this fold
    print(f"\n{'='*80}")
    print(f"GRID SEARCH WITH ALPHA OPTIMIZATION - {fold_id.upper()} (V2: NO TEST LEAKAGE)")
    print(f"{'='*80}")
    print(f"Methods to run: {methods_to_run}")
    
    all_results = []
    ALPHA_HISTORY = []
    total_experiments = len(methods_to_run) * len(K_RANGE) * len(TOPN_RANGE)
    completed = 0
    fold_start_time = time.time()
    
    for method in methods_to_run:
        # Load similarity matrix
        sim_file = os.path.join(FOLD_DIR, f"sim_{method}.npy")
        
        if not os.path.exists(sim_file):
            print(f"⚠️  Skipping {method}: similarity file not found in {fold_id}")
            continue
        
        S_user = np.load(sim_file)
        
        print(f"\n[{method.upper()}]")
        
        for k in K_RANGE:
            # ========================================
            # PHASE 1: ALPHA OPTIMIZATION (VALIDATION DATA)
            # ========================================
            # Purpose: Find optimal α using VALIDATION set
            # Data: train_inner (learning) + validation (evaluation)
            # ⚠️ TEST DATA NEVER USED IN THIS PHASE
            # ========================================
            
            print(f"  K={k}: Optimizing alpha (validation + regularization)...", end=' ')
            
            # Phase 1: Coarse search WITH REGULARIZATION
            best_coarse_alpha = None
            best_coarse_score = np.inf  # Changed from mse to score
            
            for alpha in ALPHA_COARSE_GRID:
                S_alpha = _combine_single_similarity(S_user, alpha, clip_neg=True)
                # V2: Changed to use train_inner → validation (NOT test!)
                preds_dict, mse = _knn_predict_removed_with_S(
                    X=XX_validation.values,      # V2: Validation as ground truth
                    XX=XX_train_inner.values,    # V2: Train_inner for learning
                    S=S_alpha,
                    K=k,
                    include_negative=False,
                    fallback="user_mean"
                )
                
                # V3: Apply regularization penalty
                regularization_penalty = REGULARIZATION_LAMBDA * abs(alpha - 1.0)
                regularized_score = mse + regularization_penalty
                
                # Convert predictions to DataFrame for Precision/Recall calculation
                pred_df = XX_validation.copy()  # V2: Changed from XX_test
                pred_df[:] = np.nan
                for (item_idx, user_idx), pred_rating in preds_dict.items():
                    pred_df.iloc[item_idx, user_idx] = pred_rating
                
                # Calculate Precision/Recall for ALL TopN values
                for topn in TOPN_RANGE:
                    precision, recall = precision_recall_at_n(
                        pred=pred_df,
                        train=XX_train_inner,        # V2: Changed from XX_train
                        test=XX_validation,          # V2: Changed from XX_test
                        N=topn,
                        relevance_threshold=RELEVANCE_THRESHOLD
                    )
                    
                    ALPHA_HISTORY.append({
                        'fold': fold_num,
                        'method': method,
                        'K': k,
                        'TopN': topn,  # Added TopN column
                        'phase': 'coarse',
                        'alpha': alpha,
                        'mse': mse,
                        'rmse': np.sqrt(mse),
                        'regularization_penalty': regularization_penalty,  # V3: Track penalty
                        'regularized_score': regularized_score,             # V3: Track total score
                        'precision': precision,
                        'recall': recall
                    })
                
                # V3: Use regularized score for selection
                if regularized_score < best_coarse_score:
                    best_coarse_score = regularized_score
                    best_coarse_alpha = alpha
            
            # Phase 2: Fine search WITH REGULARIZATION
            fine_start = max(0.0, best_coarse_alpha - ALPHA_FINE_RADIUS)
            fine_end = min(3.0, best_coarse_alpha + ALPHA_FINE_RADIUS)  # V3: Cap at 3.0
            fine_grid = np.arange(fine_start, fine_end + ALPHA_FINE_STEP/2, ALPHA_FINE_STEP)
            
            best_fine_alpha = best_coarse_alpha
            best_fine_score = best_coarse_score  # Changed from mse to score
            
            for alpha in fine_grid:
                S_alpha = _combine_single_similarity(S_user, alpha, clip_neg=True)
                # V2: Changed to use train_inner → validation (NOT test!)
                preds_dict, mse = _knn_predict_removed_with_S(
                    X=XX_validation.values,      # V2: Validation as ground truth
                    XX=XX_train_inner.values,    # V2: Train_inner for learning
                    S=S_alpha,
                    K=k,
                    include_negative=False,
                    fallback="user_mean"
                )
                
                # V3: Apply regularization penalty
                regularization_penalty = REGULARIZATION_LAMBDA * abs(alpha - 1.0)
                regularized_score = mse + regularization_penalty
                
                # Convert predictions to DataFrame for Precision/Recall calculation
                pred_df = XX_validation.copy()  # V2: Changed from XX_test
                pred_df[:] = np.nan
                for (item_idx, user_idx), pred_rating in preds_dict.items():
                    pred_df.iloc[item_idx, user_idx] = pred_rating
                
                # Calculate Precision/Recall for ALL TopN values
                for topn in TOPN_RANGE:
                    precision, recall = precision_recall_at_n(
                        pred=pred_df,
                        train=XX_train_inner,        # V2: Changed from XX_train
                        test=XX_validation,          # V2: Changed from XX_test
                        N=topn,
                        relevance_threshold=RELEVANCE_THRESHOLD
                    )
                    
                    ALPHA_HISTORY.append({
                        'fold': fold_num,
                        'method': method,
                        'K': k,
                        'TopN': topn,  # Added TopN column
                        'phase': 'fine',
                        'alpha': alpha,
                        'mse': mse,
                        'rmse': np.sqrt(mse),
                        'regularization_penalty': regularization_penalty,  # V3: Track penalty
                        'regularized_score': regularized_score,             # V3: Track total score
                        'precision': precision,
                        'recall': recall
                    })
                
                # V3: Use regularized score for selection
                if regularized_score < best_fine_score:
                    best_fine_score = regularized_score
                    best_fine_alpha = alpha
            
            optimal_alpha = best_fine_alpha
            print(f"α={optimal_alpha:.2f} (regularized) | Evaluating on test...", end=' ')
            
            # ========================================
            # PHASE 2: FINAL TEST EVALUATION (TEST DATA)
            # ========================================
            # Purpose: Fair comparison of optimal α vs α=1.0
            # Data: train_full (learning) + test (evaluation)
            # ✅ FIRST AND ONLY USE OF TEST DATA
            # ✅ BOTH optimal α AND α=1.0 USE SAME TEST SET
            # ========================================
            
            # 1. Optimal Alpha Evaluation
            S_alpha = _combine_single_similarity(S_user, optimal_alpha, clip_neg=True)
            preds_dict, _ = _knn_predict_removed_with_S(
                X=XX_test.values,           # V2: Test (first use!)
                XX=XX_train_full.values,    # V2: Full train for learning
                S=S_alpha,
                K=k,
                include_negative=False,
                fallback="user_mean"
            )
            
            # Convert to DataFrame
            pred_df = XX_test.copy()
            pred_df[:] = np.nan
            for (item_idx, user_idx), pred_rating in preds_dict.items():
                pred_df.iloc[item_idx, user_idx] = pred_rating
            
            # Calculate TEST RMSE and MAD (independent of TopN)
            test_rmse, test_mad = rmse_mad_on_test(pred_df, XX_test)
            
            # V4: Find VALIDATION metrics for selected alpha from history
            # This is the performance that was used to select optimal_alpha
            val_metrics_for_alpha = {}
            for topn in TOPN_RANGE:
                hist_match = [h for h in ALPHA_HISTORY if 
                             h['method'] == method and h['K'] == k and 
                             h['TopN'] == topn and h['alpha'] == optimal_alpha]
                if hist_match:
                    val_metrics_for_alpha[topn] = hist_match[-1]  # Take last (fine search)
            
            # Calculate TEST metrics for all TopN values
            for topn in TOPN_RANGE:
                test_precision, test_recall = precision_recall_at_n(
                    pred=pred_df,
                    train=XX_train_full,     # V2: Full train
                    test=XX_test,            # V2: Test
                    N=topn,
                    relevance_threshold=RELEVANCE_THRESHOLD
                )
                
                # V4: Get validation metrics for this TopN
                val_metrics = val_metrics_for_alpha.get(topn, {})
                
                all_results.append({
                    'fold': fold_num,
                    'method': method,
                    'alpha': optimal_alpha,
                    'type': 'optimized',
                    'K': k,
                    'TopN': topn,
                    # V4: VALIDATION metrics (alpha selection basis)
                    'validation_mse': val_metrics.get('mse', np.nan),
                    'validation_rmse': val_metrics.get('rmse', np.nan),
                    'validation_precision': val_metrics.get('precision', np.nan),
                    'validation_recall': val_metrics.get('recall', np.nan),
                    'regularization_penalty': val_metrics.get('regularization_penalty', np.nan),
                    'regularized_score': val_metrics.get('regularized_score', np.nan),
                    # V4: TEST metrics (generalization evaluation)
                    'test_RMSE': test_rmse,
                    'test_MAD': test_mad,
                    'test_Precision': test_precision,
                    'test_Recall': test_recall,
                    # Legacy columns for backward compatibility
                    'RMSE': test_rmse,
                    'MAD': test_mad,
                    'Precision': test_precision,
                    'Recall': test_recall
                })
                
                completed += 1

            # 2. Baseline Alpha (α=1.0) Evaluation
            # Evaluate standard similarity without optimization (alpha=1.0)
            S_one = _combine_single_similarity(S_user, 1.0, clip_neg=True)
            preds_dict_one, _ = _knn_predict_removed_with_S(
                X=XX_test.values,
                XX=XX_train_full.values,
                S=S_one,
                K=k,
                include_negative=False,
                fallback="user_mean"
            )

            # Convert to DataFrame
            pred_df_one = XX_test.copy()
            pred_df_one[:] = np.nan
            for (item_idx, user_idx), pred_rating in preds_dict_one.items():
                pred_df_one.iloc[item_idx, user_idx] = pred_rating
                
            test_rmse_baseline, test_mad_baseline = rmse_mad_on_test(pred_df_one, XX_test)
            
            for topn in TOPN_RANGE:
                test_precision_baseline, test_recall_baseline = precision_recall_at_n(
                    pred=pred_df_one,
                    train=XX_train_full,
                    test=XX_test,
                    N=topn,
                    relevance_threshold=RELEVANCE_THRESHOLD
                )
                
                # V4: For baseline, validation metrics are NaN (no validation-based selection)
                all_results.append({
                    'fold': fold_num,
                    'method': method,
                    'alpha': 1.0,
                    'type': 'baseline',
                    'K': k,
                    'TopN': topn,
                    # V4: Baseline doesn't use validation for selection
                    'validation_mse': np.nan,
                    'validation_rmse': np.nan,
                    'validation_precision': np.nan,
                    'validation_recall': np.nan,
                    'regularization_penalty': 0.0,  # No penalty for baseline
                    'regularized_score': np.nan,
                    # V4: TEST metrics
                    'test_RMSE': test_rmse_baseline,
                    'test_MAD': test_mad_baseline,
                    'test_Precision': test_precision_baseline,
                    'test_Recall': test_recall_baseline,
                    # Legacy columns
                    'RMSE': test_rmse_baseline,
                    'MAD': test_mad_baseline,
                    'Precision': test_precision_baseline,
                    'Recall': test_recall_baseline
                })
            
            # Progress update
            elapsed = time.time() - fold_start_time
            progress = (completed / total_experiments) * 100
            eta = (elapsed / completed) * (total_experiments - completed) if completed > 0 else 0
            print(f"| {progress:5.1f}% | ETA: {eta/60:.1f}min")
    
    # ========================================
    # SAVE RESULTS (with merge if applicable)
    # ========================================
    fold_time = time.time() - fold_start_time
    results_df = pd.DataFrame(all_results)
    alpha_history_df = pd.DataFrame(ALPHA_HISTORY)
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Handle merge mode
    if merge_mode and existing_results_file:
        print(f"\n{'='*80}")
        print(f"MERGING RESULTS")
        print(f"{'='*80}")
        
        output_file = os.path.join(RESULTS_DIR, f"grid_search_results_{timestamp}.csv")
        alpha_file = os.path.join(RESULTS_DIR, f"alpha_optimization_history_{timestamp}.csv")
        
        # Merge grid search results
        merge_stats = merge_results(
            new_results_df=results_df,
            existing_file=existing_results_file,
            output_file=output_file,
            create_backup=CREATE_BACKUP
        )
        
        # For alpha history, we'll merge similarly
        existing_alpha_file = get_latest_result_file(RESULTS_DIR, 'alpha_optimization_history_')
        if existing_alpha_file:
            existing_alpha_df = pd.read_csv(existing_alpha_file)
            
            # Remove existing (fold, method, K) combinations
            new_alpha_keys = alpha_history_df[['fold', 'method', 'K']].drop_duplicates()
            merged_alpha_df = existing_alpha_df.copy()
            for _, row in new_alpha_keys.iterrows():
                mask = (merged_alpha_df['fold'] == row['fold']) & \
                       (merged_alpha_df['method'] == row['method']) & \
                       (merged_alpha_df['K'] == row['K'])
                merged_alpha_df = merged_alpha_df[~mask]
            
            merged_alpha_df = pd.concat([merged_alpha_df, alpha_history_df], ignore_index=True)
            merged_alpha_df = merged_alpha_df.sort_values(['fold', 'method', 'K', 'phase', 'alpha']).reset_index(drop=True)
            merged_alpha_df.to_csv(alpha_file, index=False)
            
            if CREATE_BACKUP:
                alpha_backup = existing_alpha_file.replace('.csv', '_backup.csv')
                shutil.copy2(existing_alpha_file, alpha_backup)
        else:
            alpha_history_df.to_csv(alpha_file, index=False)
        
        # Save metadata
        experimental_conditions = {
            'K_RANGE': list(K_RANGE),
            'TOPN_RANGE': TOPN_RANGE,
            'ALPHA_COARSE_GRID': list(ALPHA_COARSE_GRID),
            'ALPHA_FINE_RADIUS': ALPHA_FINE_RADIUS,
            'ALPHA_FINE_STEP': ALPHA_FINE_STEP,
            'ALPHA_SEARCH_TOPN': ALPHA_SEARCH_TOPN,
            'RELEVANCE_THRESHOLD': RELEVANCE_THRESHOLD,
            'methods_requested': METHODS_TO_TEST,
            'methods_run': methods_to_run
        }
        
        metadata_file = save_merge_metadata(
            results_dir=RESULTS_DIR,
            merge_stats=merge_stats,
            experimental_conditions=experimental_conditions,
            timestamp=timestamp
        )
        
        print(f"\n📊 Merge Statistics:")
        print(f"  Existing rows: {merge_stats['existing_rows']:,}")
        print(f"  Existing methods: {merge_stats['existing_methods']}")
        print(f"  New rows: {merge_stats['new_rows']:,}")
        print(f"  New methods: {merge_stats['new_methods']}")
        print(f"  Merged rows: {merge_stats['merged_rows']:,}")
        print(f"  Merged methods: {merge_stats['merged_methods']}")
        if merge_stats['backup_file']:
            print(f"  Backup: {os.path.basename(merge_stats['backup_file'])}")
        
    else:
        # No merge - save as new files
        output_file = os.path.join(RESULTS_DIR, f"grid_search_results_{timestamp}.csv")
        alpha_file = os.path.join(RESULTS_DIR, f"alpha_optimization_history_{timestamp}.csv")
        
        results_df.to_csv(output_file, index=False)
        alpha_history_df.to_csv(alpha_file, index=False)
    
    print(f"\n{'='*80}")
    print(f"✅ {fold_id.upper()} COMPLETE")
    print(f"{'='*80}")
    print(f"Experiments: {completed}")
    print(f"Time: {fold_time:.1f}s ({fold_time/60:.1f} min)")
    print(f"Average: {fold_time/completed:.2f}s per experiment")
    print(f"Alpha evaluations: {len(ALPHA_HISTORY)}")
    print(f"\n💾 Saved:")
    print(f"  {output_file}")
    print(f"  {alpha_file}")
    if merge_mode:
        print(f"  {metadata_file}")
    
    # Add to global tracking
    all_folds_results.extend(all_results)
    all_folds_alpha_history.extend(ALPHA_HISTORY)

# Global summary
global_time = time.time() - global_start_time
print(f"\n{'='*80}")
print(f"🎉 ALL FOLDS COMPLETE")
print(f"{'='*80}")
print(f"Folds processed: {len(FOLDS_TO_RUN)}")
print(f"Total experiments: {len(all_folds_results)}")
print(f"Total time: {global_time:.1f}s ({global_time/60:.1f} min, {global_time/3600:.2f} hours)")
if len(FOLDS_TO_RUN) > 0:
    print(f"Average per fold: {global_time/len(FOLDS_TO_RUN)/60:.1f} min")

# Save combined results
if all_folds_results:
    combined_results_df = pd.DataFrame(all_folds_results)
    combined_alpha_df = pd.DataFrame(all_folds_alpha_history)

    combined_dir = "../results/combined"
    os.makedirs(combined_dir, exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    combined_output = os.path.join(combined_dir, f"all_folds_grid_results_{timestamp}.csv")
    combined_alpha = os.path.join(combined_dir, f"all_folds_alpha_history_{timestamp}.csv")

    combined_results_df.to_csv(combined_output, index=False)
    combined_alpha_df.to_csv(combined_alpha, index=False)

    print(f"\n💾 Combined results saved:")

    print(f"  {combined_output}")
    print(f"  {combined_output}")    
    print(f"  {combined_alpha}")
    print(f"  {combined_alpha}")


STARTING MULTI-FOLD GRID SEARCH EXPERIMENT (INCREMENTAL MODE)
Folds to process: [8, 9, 10]
Total folds: 3

FOLD 08 / 3
Directory: ../data/movielenz_data/fold_08

Loading data from: ../data/movielenz_data/fold_08
✓ Train_inner data: (1682, 943) (items × users)
  Observed ratings: 72,189
✓ Validation data: (1682, 943) (items × users)
  Observed ratings: 18,073
✓ Train_full data: (1682, 943) (items × users)
  Observed ratings: 90,262
✓ Test data: (1682, 943) (items × users)
  Observed ratings: 9,738

GRID SEARCH WITH ALPHA OPTIMIZATION - FOLD_08 (V2: NO TEST LEAKAGE)
Methods to run: ['pcc', 'cosine', 'msd', 'jmsd', 'jaccard', 'spcc', 'kendall_tau_b', 'acos', 'src', 'cpcc', 'ari', 'ami', 'chebyshev', 'euclidean', 'manhattan', 'ipwr', 'itr']

[PCC]
  K=10: Optimizing alpha (validation + regularization)... 

/var/folders/p3/9w5k2wns08dg9020pszxby600000gn/T/ipykernel_43168/2602842477.py:21: RuntimeWarning: Mean of empty slice
  item_means = np.nanmean(XX, axis=1)


α=2.35 (regularized) | Evaluating on test... |   0.6% | ETA: 311.5min
  K=20: Optimizing alpha (validation + regularization)... α=3.00 (regularized) | Evaluating on test... |   1.2% | ETA: 274.1min
  K=30: Optimizing alpha (validation + regularization)... α=3.50 (regularized) | Evaluating on test... |   1.8% | ETA: 237.3min
  K=40: Optimizing alpha (validation + regularization)... α=4.00 (regularized) | Evaluating on test... |   2.4% | ETA: 216.6min
  K=50: Optimizing alpha (validation + regularization)... α=4.00 (regularized) | Evaluating on test... |   2.9% | ETA: 203.8min
  K=60: Optimizing alpha (validation + regularization)... α=4.00 (regularized) | Evaluating on test... |   3.5% | ETA: 195.0min
  K=70: Optimizing alpha (validation + regularization)... α=4.50 (regularized) | Evaluating on test... |   4.1% | ETA: 188.5min
  K=80: Optimizing alpha (validation + regularization)... α=4.50 (regularized) | Evaluating on test... |   4.7% | ETA: 183.3min
  K=90: Optimizing alpha (validati

/var/folders/p3/9w5k2wns08dg9020pszxby600000gn/T/ipykernel_43168/2602842477.py:21: RuntimeWarning: Mean of empty slice
  item_means = np.nanmean(XX, axis=1)


α=2.50 (regularized) | Evaluating on test... |   0.6% | ETA: 306.9min
  K=20: Optimizing alpha (validation + regularization)... α=3.50 (regularized) | Evaluating on test... |   1.2% | ETA: 235.5min
  K=30: Optimizing alpha (validation + regularization)... α=4.00 (regularized) | Evaluating on test... |   1.8% | ETA: 208.9min
  K=40: Optimizing alpha (validation + regularization)... α=4.00 (regularized) | Evaluating on test... |   2.4% | ETA: 195.3min
  K=50: Optimizing alpha (validation + regularization)... α=4.50 (regularized) | Evaluating on test... |   2.9% | ETA: 186.7min
  K=60: Optimizing alpha (validation + regularization)... α=4.50 (regularized) | Evaluating on test... |   3.5% | ETA: 180.7min
  K=70: Optimizing alpha (validation + regularization)... α=4.50 (regularized) | Evaluating on test... |   4.1% | ETA: 176.1min
  K=80: Optimizing alpha (validation + regularization)... α=4.50 (regularized) | Evaluating on test... |   4.7% | ETA: 172.5min
  K=90: Optimizing alpha (validati

/var/folders/p3/9w5k2wns08dg9020pszxby600000gn/T/ipykernel_43168/2602842477.py:21: RuntimeWarning: Mean of empty slice
  item_means = np.nanmean(XX, axis=1)


α=2.55 (regularized) | Evaluating on test... |   0.6% | ETA: 307.6min
  K=20: Optimizing alpha (validation + regularization)... α=3.50 (regularized) | Evaluating on test... |   1.2% | ETA: 236.3min
  K=30: Optimizing alpha (validation + regularization)... α=4.00 (regularized) | Evaluating on test... |   1.8% | ETA: 209.6min
  K=40: Optimizing alpha (validation + regularization)... α=4.00 (regularized) | Evaluating on test... |   2.4% | ETA: 196.0min
  K=50: Optimizing alpha (validation + regularization)... α=4.00 (regularized) | Evaluating on test... |   2.9% | ETA: 187.3min
  K=60: Optimizing alpha (validation + regularization)... α=4.50 (regularized) | Evaluating on test... |   3.5% | ETA: 181.1min
  K=70: Optimizing alpha (validation + regularization)... α=4.50 (regularized) | Evaluating on test... |   4.1% | ETA: 176.6min
  K=80: Optimizing alpha (validation + regularization)... α=4.50 (regularized) | Evaluating on test... |   4.7% | ETA: 172.9min
  K=90: Optimizing alpha (validati

In [16]:
# ========================================
# Step 7.5: Multi-Lambda Integration
# (Run this AFTER Step 7 performs the main heavy lifting)
# ========================================

# 1. 추가 분석할 Lambda 리스트 정의
ADDITIONAL_LAMBDAS = [0.001, 0.003, 0.004, 0.005, 0.01, 0.015, 0.02, 0.025, 0.03]
print(f"\n{'='*80}")
print(f"STEP 7.5: MULTI-LAMBDA INTEGRATION")
print(f"{'='*80}")
print(f"Processing additional lambdas: {ADDITIONAL_LAMBDAS}")

# ========================================
# Load Required Data (From Memory or Disk)
# ========================================

history_list = []
results_list = []
processed_folds = []

# Check memory first
if 'all_folds_alpha_history' in locals() and len(all_folds_alpha_history) > 0 and \
   'all_folds_results' in locals() and len(all_folds_results) > 0:
    print("✅ Using data from memory (Step 7 results)")
    history_df = pd.DataFrame(all_folds_alpha_history)
    step7_df = pd.DataFrame(all_folds_results)
    
    loaded_folds = sorted(step7_df['fold'].unique())
    print(f"  Loaded folds from memory: {loaded_folds}")
    
else:
    print("⚠️  Loading latest data from INDIVIDUAL FOLD directories...")
    
    base_results_dir = "../results"
    
    # Determine which folds to load
    # PRIORITY: Use FOLDS_TO_RUN from Step 2 configuration
    if 'FOLDS_TO_RUN' in locals() and FOLDS_TO_RUN:
        target_folds = FOLDS_TO_RUN if isinstance(FOLDS_TO_RUN, list) else list(FOLDS_TO_RUN)
        print(f"  Target folds (from FOLDS_TO_RUN): {target_folds}")
    else:
        # Fallback: scan all folds (1-10)
        target_folds = list(range(1, 11))
        print(f"  ℹ️  FOLDS_TO_RUN not defined - scanning all folds")
    
    # Load only target folds
    for fold_num in target_folds:
        fold_id = f"fold_{fold_num:02d}"
        fold_dir = os.path.join(base_results_dir, fold_id)
        
        if not os.path.exists(fold_dir):
            print(f"  ⚠️  {fold_id} directory not found - skipping")
            continue
            
        # Find latest files in this fold
        grid_file = get_latest_result_file(fold_dir, 'grid_search_results_')
        alpha_file = get_latest_result_file(fold_dir, 'alpha_optimization_history_')
        
        if grid_file and alpha_file:
            print(f"  ✓ {fold_id}: {os.path.basename(grid_file)}")
            
            # Load and append
            try:
                f_res = pd.read_csv(grid_file)
                f_hist = pd.read_csv(alpha_file)
                
                results_list.append(f_res)
                history_list.append(f_hist)
                processed_folds.append(fold_num)
            except Exception as e:
                print(f"    ❌ Error loading {fold_id}: {e}")
        else:
            print(f"  ⚠️  {fold_id}: Result files not found - skipping")
    
    if not results_list:
        raise FileNotFoundError(f"No result files found for target folds: {target_folds}")
        
    step7_df = pd.concat(results_list, ignore_index=True)
    history_df = pd.concat(history_list, ignore_index=True)
    print(f"  ✅ Loaded {len(step7_df):,} rows from folds: {processed_folds}")

# ========================================
# Determine Processing Scope
# ========================================

available_folds = sorted(step7_df['fold'].unique().tolist())

# Use FOLDS_TO_RUN if available, otherwise use what we loaded
if 'FOLDS_TO_RUN' in locals() and isinstance(FOLDS_TO_RUN, list):
    requested_folds = FOLDS_TO_RUN if isinstance(FOLDS_TO_RUN, list) else list(FOLDS_TO_RUN)
    common_folds = [f for f in requested_folds if f in available_folds]
    
    if len(common_folds) > 0:
        FOLDS_TO_RUN = common_folds
        print(f"\n✅ Processing folds from FOLDS_TO_RUN: {FOLDS_TO_RUN}")
    else:
        print(f"\n⚠️  Requested folds {requested_folds} not in loaded data {available_folds}")
        FOLDS_TO_RUN = available_folds
        print(f"    Fallback: Using all loaded folds: {FOLDS_TO_RUN}")
else:
    FOLDS_TO_RUN = available_folds
    print(f"\nℹ️  FOLDS_TO_RUN not defined - using all loaded folds: {FOLDS_TO_RUN}")

# Verify required variables
if 'K_RANGE' not in locals():
    K_RANGE = sorted(history_df['K'].unique().tolist())
if 'TOPN_RANGE' not in locals():
    TOPN_RANGE = sorted(history_df['TopN'].unique().tolist())
if 'RELEVANCE_THRESHOLD' not in locals():
    RELEVANCE_THRESHOLD = 4.0
if 'REGULARIZATION_LAMBDA' not in locals():
    REGULARIZATION_LAMBDA = 0.002

print(f"\n📊 Processing Summary:")
print(f"  Folds: {FOLDS_TO_RUN}")
print(f"  K values: {len(K_RANGE)} values")
print(f"  TopN values: {len(TOPN_RANGE)} values")
print(f"  Additional lambdas: {len(ADDITIONAL_LAMBDAS)} values")

# Filter history to only include folds we're processing
history_df = history_df[history_df['fold'].isin(FOLDS_TO_RUN)]
print(f"  Filtered history: {len(history_df):,} rows")

# ========================================
# Process Additional Lambdas
# ========================================

additional_results = []

for fold_num in FOLDS_TO_RUN:
    fold_id = f"fold_{fold_num:02d}"
    FOLD_DIR = f"../data/movielenz_data/{fold_id}"
    TRAIN_CSV = os.path.join(FOLD_DIR, "train.csv")
    TEST_CSV = os.path.join(FOLD_DIR, "test.csv")
    
    print(f"\n{'='*80}")
    print(f"Processing {fold_id} for additional lambdas...")
    print(f"{'='*80}")
    
    try:
        XX_train_full = _load_matrix_csv(TRAIN_CSV)
        XX_test = _load_matrix_csv(TEST_CSV)
        print(f"✓ Data loaded: Train {XX_train_full.shape}, Test {XX_test.shape}")
    except Exception as e:
        print(f"❌ Failed to load data for {fold_id}: {e}")
        continue
    
    fold_history = history_df[history_df['fold'] == fold_num]
    methods_in_fold = sorted(fold_history['method'].unique())
    
    print(f"Methods ({len(methods_in_fold)}): {methods_in_fold}")
    
    for method in methods_in_fold:
        print(f"  [{method}]", end=" ")
        
        sim_file = os.path.join(FOLD_DIR, f"sim_{method}.npy")
        if not os.path.exists(sim_file):
            print("(sim file not found)")
            continue
        
        S_user = np.load(sim_file)
        
        for k in K_RANGE:
            print(".", end="", flush=True)
            
            subset = fold_history[
                (fold_history['method'] == method) & 
                (fold_history['K'] == k) & 
                (fold_history['phase'].isin(['coarse', 'fine']))
            ].copy()
            
            if subset.empty: continue
            
            for new_lambda in ADDITIONAL_LAMBDAS:
                # 1. Re-optimize Alpha
                alpha_metrics = subset[['alpha', 'mse']].drop_duplicates()
                alpha_metrics['reg_score'] = alpha_metrics['mse'] + new_lambda * np.abs(alpha_metrics['alpha'] - 1.0)
                best_row = alpha_metrics.loc[alpha_metrics['reg_score'].idxmin()]
                optimal_alpha = best_row['alpha']
                
                # 2. Test Evaluation
                S_alpha = _combine_single_similarity(S_user, optimal_alpha, clip_neg=True)
                preds_dict, _ = _knn_predict_removed_with_S(
                    X=XX_test.values, XX=XX_train_full.values, S=S_alpha,
                    K=k, include_negative=False, fallback="user_mean"
                )
                
                pred_df = XX_test.copy(); pred_df[:] = np.nan
                for (item_idx, user_idx), pred_rating in preds_dict.items():
                    pred_df.iloc[item_idx, user_idx] = pred_rating
                
                test_rmse, test_mad = rmse_mad_on_test(pred_df, XX_test)
                
                for topn in TOPN_RANGE:
                    val_rec = subset[(subset['alpha'] == optimal_alpha) & (subset['TopN'] == topn)]
                    if not val_rec.empty:
                        val_mse = val_rec.iloc[0]['mse']
                        val_rmse = val_rec.iloc[0]['rmse']
                        val_prec = val_rec.iloc[0]['precision']
                        val_rec_val = val_rec.iloc[0]['recall']
                    else:
                        val_mse = np.nan; val_rmse = np.nan; val_prec = np.nan; val_rec_val = np.nan

                    test_precision, test_recall = precision_recall_at_n(
                        pred=pred_df, train=XX_train_full, test=XX_test,
                        N=topn, relevance_threshold=RELEVANCE_THRESHOLD
                    )
                    
                    additional_results.append({
                        'fold': fold_num, 'method': method,
                        'alpha': optimal_alpha, 'type': 'optimized',
                        'K': k, 'TopN': topn, 'lambda': new_lambda,
                        'validation_mse': val_mse, 'validation_rmse': val_rmse,
                        'validation_precision': val_prec, 'validation_recall': val_rec_val,
                        'regularization_penalty': new_lambda * abs(optimal_alpha - 1.0),
                        'regularized_score': val_mse + new_lambda * abs(optimal_alpha - 1.0),
                        'test_RMSE': test_rmse, 'test_MAD': test_mad,
                        'test_Precision': test_precision, 'test_Recall': test_recall,
                        'RMSE': test_rmse, 'MAD': test_mad, 'Precision': test_precision, 'Recall': test_recall
                    })
        print(" ✓")

# ========================================
# 3. 통합 및 저장 (Merge)
# ========================================

print(f"\n{'='*80}")
print(f"MERGING AND SAVING")
print(f"{'='*80}")

if 'lambda' not in step7_df.columns:
    step7_df['lambda'] = REGULARIZATION_LAMBDA

additional_df = pd.DataFrame(additional_results)
current_batch_df = pd.concat([step7_df, additional_df], ignore_index=True)

print(f"Current batch: {len(current_batch_df):,} rows")
print(f"  - Step 7 (λ={REGULARIZATION_LAMBDA}): {len(step7_df):,} rows")
print(f"  - Additional lambdas: {len(additional_df):,} rows")

# Load existing integrated file for merging
combined_dir = "../results/combined"
existing_files = [f for f in os.listdir(combined_dir) 
                  if f.startswith('integrated_lambda_grid_results_') and f.endswith('.csv')]

merged_df = current_batch_df

if existing_files:
    latest_path = os.path.join(combined_dir, sorted(existing_files)[-1])
    print(f"\n📂 Found existing: {os.path.basename(latest_path)}")
    
    try:
        existing_df = pd.read_csv(latest_path)
        print(f"  Loaded: {len(existing_df):,} rows")
        
        # Remove overlapping (fold, method) from existing
        new_keys = current_batch_df[['fold', 'method']].drop_duplicates()
        keys_to_remove = set(zip(new_keys['fold'], new_keys['method']))
        
        mask = existing_df.apply(lambda x: (x['fold'], x['method']) in keys_to_remove, axis=1)
        filtered_existing = existing_df[~mask]
        
        removed_count = len(existing_df) - len(filtered_existing)
        print(f"  Removed {removed_count:,} overlapping rows")
        
        merged_df = pd.concat([filtered_existing, current_batch_df], ignore_index=True)
        print(f"  Merged total: {len(merged_df):,} rows")
    except Exception as e:
        print(f"⚠️ Merge error: {e} - saving current batch only")

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = f"../results/combined/integrated_lambda_grid_results_{timestamp}.csv"

sort_cols = [c for c in ['fold', 'method', 'K', 'TopN', 'lambda', 'type'] if c in merged_df.columns]
if sort_cols: merged_df = merged_df.sort_values(sort_cols)

merged_df.to_csv(output_path, index=False)

print(f"\n{'='*80}")
print(f"✅ INTEGRATION COMPLETE")
print(f"{'='*80}")
print(f"Final dataset: {len(merged_df):,} rows")
print(f"Folds: {sorted(merged_df['fold'].unique())}")
print(f"Lambdas: {sorted(merged_df['lambda'].unique())}")
print(f"\n💾 Saved to:")
print(f"  {output_path}")



STEP 7.5: MULTI-LAMBDA INTEGRATION
Processing additional lambdas: [0.001, 0.003, 0.004, 0.005, 0.01, 0.015, 0.02, 0.025, 0.03]
✅ Using data from memory (Step 7 results)
  Loaded folds from memory: [np.int64(8), np.int64(9), np.int64(10)]

✅ Processing folds from FOLDS_TO_RUN: [8, 9, 10]

📊 Processing Summary:
  Folds: [8, 9, 10]
  K values: 10 values
  TopN values: 10 values
  Additional lambdas: 9 values
  Filtered history: 165,840 rows

Processing fold_08 for additional lambdas...
✓ Data loaded: Train (1682, 943), Test (1682, 943)
Methods (17): ['acos', 'ami', 'ari', 'chebyshev', 'cosine', 'cpcc', 'euclidean', 'ipwr', 'itr', 'jaccard', 'jmsd', 'kendall_tau_b', 'manhattan', 'msd', 'pcc', 'spcc', 'src']
  [acos] .......... ✓
  [ami] .......... ✓
  [ari] .......... ✓
  [chebyshev] .......... ✓
  [cosine] .......... ✓
  [cpcc] .......... ✓
  [euclidean] .......... ✓
  [ipwr] .......... ✓
  [itr] .......... ✓
  [jaccard] .......... ✓
  [jmsd] .......... ✓
  [kendall_tau_b] .......... ✓
 